# Titanitron

This cell imports dependencies 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
# Load the datasets
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
train_df.head()

There is some missing information in  the train dataset, so this next cell displays them.

In [ ]:
  # Basic information about the dataset
print("Dataset Info:")
print(train_df.info())
print("\nMissing values:")
print(train_df.isnull().sum())
print("\nSurvival statistics:")
print(train_df['Survived'].value_counts())
print(f"Survival rate: {train_df['Survived'].mean():.3f}")

Since the passengerid and ticket columns are unneccessary, this next cell removes them from the dataframe.

In [ ]:
train_df.drop(columns=['PassengerId', 'Ticket'], inplace=True)
train_df.info()
train_df.head(10)

This cell fills the missing embarked values with the mode of the embarked category.

In [ ]:
train_df['Embarked'].fillna(train_df['Embarked'].mode()[0], inplace=True)
train_df.isnull().sum()

To fill the missing values for age, each missing value is filled with the median value for each respective gender and class.

In [ ]:
train_df['Age'] = train_df.groupby(['Pclass', 'Sex'])['Age'].transform(
lambda x: x.fillna(x.median())
)

After plotting training data, we found a pattern where younger people had a higher survival rate.

In [ ]:
# Age analysis show that young people have higher survival rate.
# Survival percentage by meaningful age groups
train_viz2 = train_df.copy()
train_viz2['Age_Group'] = pd.cut(train_viz2['Age'],

bins=[0, 12, 18, 30, 50, 80],
labels=['Child (0-12)', 'Teen (13-18)', 'Young Adult (19-30)',
'Adult (31-50)', 'Senior (50+)'])
sns.barplot(data=train_viz2, x='Age_Group', y='Survived', ci=None)
plt.title('Survival Percentage by Age Groups')
plt.xlabel('Age Group')
plt.ylabel('Survival Rate')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Since the cabin category is missing too much data, it is dropped from the dataset.

In [ ]:
train_df.drop(columns=['Cabin'], inplace=True)

Another pattern that we found was the people with a higher social status 

In [ ]:
sns.countplot(data=train_df, x='Pclass', hue='Survived')
plt.title('Survival by Class')
plt.xlabel('Passenger Class')
plt.ylabel('Count')
plt.show()

Ordinally encodes the class category

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
# Ordinal encoding for ordinal categories
encoder = OrdinalEncoder(categories=[['1', '2', '3']])
train_df['Pclass_encoded'] = encoder.fit_transform(train_df[['Pclass']])
train_df.drop(columns=['Pclass'], inplace=True)
train_df.head(10)

After analyzing this graph, we realized that females were more likely to survive the Titanic.

In [ ]:
sns.countplot(data=train_df, x='Sex', hue='Survived')
plt.title('Survival by Gender')
plt.show()

This encodes the sex category with one hot encoding.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, dtype=int)
sex_encoded_array = encoder.fit_transform(train_df[['Sex']])
sex_columns = encoder.get_feature_names_out(['Sex'])
sex_encoded = pd.DataFrame(sex_encoded_array, columns=sex_columns, index=train_df.index)
sex_encoded.head(10)

The sex category is then combined with the main dataframe.

In [ ]:
train_df = pd.concat([train_df, sex_encoded], axis=1)
train_df.head(10)

The old sex category is now removed

In [ ]:
train_df = pd.concat([train_df, sex_encoded], axis=1)
train_df.head(10)

Now, the embarked feature is also encoded using one hot encoding.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)
embarked_encoded_array = encoder.fit_transform(train_df[['Embarked']])
embarked_columns = encoder.get_feature_names_out(['Embarked'])
embarked_encoded = pd.DataFrame(embarked_encoded_array,␣
↪columns=embarked_columns, index=train_df.index)
# combine new embarked columns with original dataframe
train_df = pd.concat([train_df, embarked_encoded], axis=1)

11

# remove old embarked column
train_df.drop(columns=['Embarked'], inplace=True)
train_df.head(10)